# Week 5 へようこそ - エージェントフレームワーク

## Day 3: Agno

同じ週、同じ5つのステップ、新しいフレームワークです。今週の考え方全体は、ひとつのエージェントフレームワークを理解すれば他もだいたい理解できるということなので、毎日同じ5つのステップで同じエージェントを作り、その作法が響き合う様子を観察します。

1. **エージェントを作る** - モデルとシステムプロンプトを与える。
2. **実行する** - メッセージを送り、返信を受け取る。
3. **ツールを追加する** - エージェントが呼び出せる、普通の型付き関数。
4. **MCP を追加する** - 誰か他の人が書いたツールサーバーに接続する。毎回同じ方法でつなげる。
5. **ゴールを与えてループさせる** - 目標を渡し、仕事が終わるまで一歩ずつ自分で進めさせる。

ステップ1と2は、まだ単なる LLM 呼び出しです。ツールと MCP は、エージェントにできることを与えます。ステップ5でようやくエージェントらしくなります。フレームワーク自身がループを回し、ツールを選び、結果を読み、また選び直す、というのをゴールに到達するまで続けるのです。

実習プロジェクトは Day 1 と同じ SQLite の todo ボードです。ワーカーがボードから1つのゴールを取り出し、自分でステップを計画し、自分のエージェントループでその作業をこなし、各ステップにチェックを入れていきます。ボードのコード(`board.py`)は一字一句まったく同じファイルで、変わるのはそれを取り巻くフレームワークだけです。

今日は **Agno** です。かつて Phidata の開発チームだった人たちによるもので、彼らはこれを「エージェント型ソフトウェアのためのプログラミング言語」と呼んでいます。特に目立つ点が2つあります。インスタンス化が驚くほど速く軽量であること、そしてここで書いたエージェントがそのまま変更なしに、セッション、トレース、コントロールプレーン UI を備えた独自の FastAPI ランタイムである AgentOS にそのまま乗せられることです。ここでは素の SDK を使い、最後にインスタンス化の速さを計測します。他のフレームワークとの違いがどれだけ少ないかに注目してください。それこそが伝えたいポイントです。

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Agno のドキュメント</h2>
            <span style="color:#00bfff;">ドキュメントは <a href="https://docs.agno.com/introduction">docs.agno.com</a> にあります。Agno の v2(AgentOS への書き換え)では v1 からかなり多くの名前が変わったので、ここではバージョンを固定(2.6.14)し、古い Phidata 時代の記事より最新のページを優先します。モデルは <code>OpenAIChat(id=...)</code> で、エージェントは <code>Agent(model=..., tools=[...])</code> です。1つ知っておくべきこととして、MCP を接続するとエージェントが非同期で動くようになるので、<code>arun</code> で <code>await</code> します。</span>
        </td>
    </tr>
</table>

## セットアップ

今日必要なものは2つですが、どちらも以前の週からすでに用意されています。

- **Node**。`npx` のために必要です(filesystem MCP サーバーはこれを使って動きます)。`node --version` で確認してください。
- リポジトリのルートにある `.env` の中の **`GOOGLE_API_KEY`**。今日のフレームワークは Gemini の `gemini-flash-latest` を、GeminiのOpenAI互換エンドポイント経由で使います。

Agno はリポジトリの環境に含まれているので、リポジトリのルートで通常の `uv sync` を実行すればすべてインストールされます。このノートブックを Cursor で開き、毎週使っているリポジトリ既定の **Python 3.12.12** カーネルを選んで、上から順にセルを実行してください。

最初の実行を速くするために、今のうちに一度 filesystem MCP サーバーをウォームアップしておき、動作中と表示されたらすぐに Ctrl-C で止めてください。

```bash
npx -y @modelcontextprotocol/server-filesystem .
```

In [ ]:
import functools
import os
import subprocess
from pathlib import Path

from dotenv import load_dotenv
from agno.agent import Agent
from agno.models.openai.like import OpenAILike
from agno.tools.mcp import MCPTools
from mcp import StdioServerParameters

load_dotenv(override=True)

## ステップ1: エージェントを作る

Agno では、エージェントは `Agent` です。`model` と、システムプロンプトである `instructions` を持ちます。ここではモデルを、GeminiのOpenAI互換エンドポイントを指す `OpenAILike` として一度だけ組み立て、環境変数から `GOOGLE_API_KEY` を読み込み、それを再利用します。Agno には同期的な経路もありますが、後で MCP を接続するとエージェントが非同期になるため、ノートブック全体で一貫した見え方になるよう、ずっと `arun` で `await` します。

In [ ]:
MODEL = "gemini-flash-latest"

# メインモデルをOpenAIからGeminiに切り替え(GeminiのOpenAI互換エンドポイント経由)
model = OpenAILike(
    id=MODEL,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.environ["GOOGLE_API_KEY"],
)

agent = Agent(
    model=model,
    instructions="You are a concise, friendly assistant. Reply in a single short sentence.",
)

## ステップ2: 実行する

メッセージを送り、返信を待ち、結果の `.content` を表示します。まだツールがないので、ループするものは何もなく、エージェントはただ答えるだけです。これはまだ単なる LLM 呼び出しです。

In [ ]:
result = await agent.arun(input="Say hello in Spanish.")
print(result.content)

## 今週のプロジェクト: SQLite の todo ボード

ワーカーは、Day 1 と同じ小さな SQLite ボード、同じ `board.py` ファイルを介して連携します。1つのファイル、1つのテーブルで、サーバーを立てる必要もありません。ワーカーには1つの**ゴール**が与えられ、それを達成するために自分自身の**ステップ**の todo をそのゴールの下に書き出し、進めるごとにチェックを入れていき、最後にゴールを完了にします。内部的にはボードは単なる辞書のリストです(ゴールの `parent_id` は None で、ステップは自分のゴールを指します)。

In [ ]:
import board

board.reset_board()
board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.")
board.list_todos()

`show_board()` は、Week 1 で使ったのと同じ rich スタイルで、その同じデータを綺麗に表示します。各ゴールの下にステップがインデントされて並び、完了したタスクは緑色の打ち消し線、進行中のタスクは黄色で表示されます。まだステップはありません。エージェントが計画を立てるときに自分でステップを書き出します。

In [ ]:
board.show_board()

## ステップ3: ツールを追加する

Agno でのツールは、docstring 付きの普通の型付き Python 関数です。Agno が型ヒントと docstring を読み取り、スキーマを自動で組み立ててくれるので、他に宣言することは何もありません。任意で使える `@tool` デコレーターもありますが、それはキャッシングなどの追加機能のためだけに存在します。関数はエージェントの `tools=[...]` リストに渡します。

ここでは3つの小さなボードツールを書きます。ボードを読む `show_todos`、ゴールをステップに分解する `plan_steps`、todo を完了にする `complete_task` です。まずは簡単なエージェントに2つだけ与えて、ボードに何があるか尋ねてみましょう。答える前に自分から `show_todos` を呼び出すことを、自分の目で確かめてください。この「決める、呼ぶ、読む、答える」というサイクルこそ、エージェントループが回り始めた瞬間です。3つのツールすべてはステップ5で一緒になります。

In [ ]:
def show_todos() -> list[dict]:
    """List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board. Pass the goal's id and a short list of step descriptions."""
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}

def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a step or the goal) with this id as done and record a short result summary."""
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "done"}

In [ ]:
board_agent = Agent(
    model=model,
    instructions="You help manage a shared todo board.",
    tools=[show_todos, complete_task],
)

In [ ]:
result = await board_agent.arun(input="What is on the board right now, and what is its status?")
print(result.content)

## ステップ4: MCP を追加する

MCP は、単に「自分が書いていないツール」を、小さなプロトコル越しに接続したものです。今週すべてのフレームワークで使う同じ Node サーバーである filesystem リファレンスサーバーを、単一の `workspace` フォルダに限定してエージェントに与えます。これにより、エージェントはそのフォルダ内のファイルしか触れなくなります。Agno では、MCP サーバーは `MCPTools` という非同期コンテキストマネージャで、`async with` で開いて、同じ `tools=[...]` リストでエージェントに渡します。`StdioServerParameters` から組み立てることで、サーバーの作業ディレクトリ(`cwd`)を workspace に設定できるので、エージェントのファイル名はそこで解決されるようになります。

Agno はサーバーの stderr を公開していないので、次のセルの最初の行では、その stdio クライアントをヌルデバイスに向けています。これによりサーバーの起動時バナーが静かになり、Windows 上の Jupyter カーネルからサーバーを実行できるようになります。Windows のカーネルの stderr には実際のファイルディスクリプタがないためです。Mac と Linux では、単純に出力が綺麗になるだけです。

In [ ]:
# Agno は MCP サーバーの stderr を公開していないので、その stdio クライアントを DEVNULL に向ける。
import agno.tools.mcp.mcp as agno_mcp
agno_mcp.stdio_client = functools.partial(agno_mcp.stdio_client, errlog=subprocess.DEVNULL)

workspace = Path("workspace").resolve()   # エージェントが触れてよい唯一のフォルダ

server = StdioServerParameters(
    command="npx",
    args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
    cwd=str(workspace),
)

In [ ]:
async with MCPTools(server_params=server, timeout_seconds=60) as filesystem:
    file_agent = Agent(
        model=model,
        instructions="You can read and write files in your workspace. Use your tools to do what is asked.",
        tools=[filesystem],
    )
    result = await file_agent.arun(input="Read notes.txt and summarize it in one short sentence.")
print(result.content)

## ステップ5: ゴールを与えてループさせる

さあ、いよいよ本番です。1つのエージェントに3つのボードツールすべてと filesystem サーバーを与え、ゴールを渡して、実行させましょう。エージェントは自分でボード上にステップを計画し、ファイルツールでそれを片付け、それぞれにチェックを入れ、作業が終わったらゴールを完了にします。これこそ、自律的に動くエージェントループです。読む、計画する、行動する、チェックする、繰り返す。ボードにステップが埋まり、それが打ち消し線で消されていく様子を観察してください。

In [ ]:
INSTRUCTIONS = """
You are a careful worker with a shared todo board and a set of file tools.

Take the pending goal and see it through. Begin by laying out a short plan: the handful of concrete steps the work itself breaks down into, added to the board under the goal. Then carry them out with your file tools, marking each step done as you finish it. Once the steps are all done, close the goal. Your files live in the single folder your tools are allowed to use.
"""

board.reset_board()
goal_id = board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.")
board.claim_todo(goal_id)

async with MCPTools(server_params=server, timeout_seconds=60) as filesystem:
    worker = Agent(
        model=model,
        instructions=INSTRUCTIONS,
        tools=[show_todos, plan_steps, complete_task, filesystem],
    )
    await worker.arun(input="Please work the pending goal on the board.")
board.show_board()

## 同じワーカーをターミナルから実行する

ステップ5でたった今観察した内容はすべて、このノートブックの隣にある小さなスクリプト `agno_worker.py` としてもパッケージ化されています。これは同じゴールを登録し、同じ3つのボードツールと filesystem MCP サーバーを使って同じエージェントを組み立て、カーネルではなくコマンドラインから同じループを実行します。このフォルダでターミナルを開いて実行してください。

```bash
uv run agno_worker.py
```

エージェントがステップを計画し、ゴールに取り組んでそれぞれにチェックを入れていく様子、そして完成したボードと、書き出されたスペイン語が表示されます。これは Day 5 でどのワーカーも取る形と同じです。Day 5 では、Google ADK のオーケストレーターが、フレームワークごとにこうしたワーカーを1つずつ、共有された1つのボードに対する並列サブプロセスとして起動します。

## いちばん興味深い点: 起動の速さ

Agno の目玉は速さです。エージェントの生成はほぼ無料と言えるほど速く、サーバーがリクエストごとに新しいエージェントを立ち上げたり、チームが一度に多数を生成したりする場面で重要になります。実際に計測してみましょう。エージェントのバッチを作って時間を測ります。

```
(数値は環境によって変わりますが、各エージェントはおよそ数マイクロ秒に収まります)
```

In [ ]:
import time

start = time.perf_counter()
agents = [Agent(model=model, tools=[show_todos]) for _ in range(1000)]
elapsed = time.perf_counter() - start
print(f"Built {len(agents)} agents in {elapsed * 1000:.1f} ms")
print(f"That is about {elapsed / len(agents) * 1_000_000:.1f} microseconds each.")

もう半分の話は **AgentOS** です。ほんの数行で、あなたのエージェントをセッション、ストリーミング、トレース、コントロールプレーン UI を備えた FastAPI アプリでラップできるので、今組み立てたばかりのエージェントを、変更なしに本番環境で提供できます。そして、今週の他の部分と同じワンライン切り替えの精神で、Agno は `OpenAILike(id=..., base_url=..., api_key=...)` を通じて任意の OpenAI 互換エンドポイントと話すことができ、ツール、MCP サーバー、ボードはすべて変わりません。

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">エクササイズ</h2>
            <span style="color:#ff7800;">ボードに別のゴールを、たとえば「マドリードについての短い俳句を書いて madrid.txt に保存する」を登録し、ワーカーを再度実行してみましょう。ワーカーは適切なステップを計画し、正しいファイルツールを選べるでしょうか。次に、<code>arun</code> の代わりに <code>await agent.aprint_response(input=..., stream=True)</code> でエージェントを実行し、Agno がツール呼び出しのループをその場でリアルタイムに表示する様子を観察してみましょう。</span>
        </td>
    </tr>
</table>